<a href="https://colab.research.google.com/github/narakrm/northstar_databases_analytics/blob/main/Section4_SQL_in_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 4 — SQL in R
**NorthStar Urban Mobility and Logistics — Databases and Analytics Assignment**

This notebook uses RSQLite to load the NorthStar CSV files into an in-memory relational database and execute SQL queries that investigate the key business problems identified in Section 2.

**Queries covered:**
1. Zone-level delivery failure rates
2. Complaint vs completion mismatch
3. Manual route override analysis by zone
4. Profitability by service type
5. Driver performance ranking

---
> **Note:** Upload all NorthStar CSV files to `/content/` before running this notebook,  
> or mount Google Drive and adjust the `data_path` variable in Cell 1.

## Cell 1 — Install packages and load libraries

In [ ]:
# Install required packages (only needed once per Colab session)
install.packages(c("RSQLite", "DBI", "dplyr", "knitr"), quiet = TRUE)

library(RSQLite)
library(DBI)
library(dplyr)
library(knitr)

cat("Packages loaded successfully\n")


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




Packages loaded successfully


## Cell 2 — Load and clean CSV files

In [ ]:
# Set data path — change this if using Google Drive
data_path <- "/content/"

# Load all CSV files
orders      <- read.csv(paste0(data_path, "orders.csv"),      stringsAsFactors = FALSE)
deliveries  <- read.csv(paste0(data_path, "deliveries.csv"),  stringsAsFactors = FALSE)
complaints  <- read.csv(paste0(data_path, "complaints.csv"),  stringsAsFactors = FALSE)
drivers     <- read.csv(paste0(data_path, "drivers.csv"),     stringsAsFactors = FALSE)
customers   <- read.csv(paste0(data_path, "customers.csv"),   stringsAsFactors = FALSE)
vehicles    <- read.csv(paste0(data_path, "vehicles.csv"),    stringsAsFactors = FALSE)
hubs        <- read.csv(paste0(data_path, "hubs.csv"),        stringsAsFactors = FALSE)
incidents   <- read.csv(paste0(data_path, "incidents.csv"),   stringsAsFactors = FALSE)
app_events  <- read.csv(paste0(data_path, "app_events.csv"),  stringsAsFactors = FALSE)

# ── Zone standardisation ──────────────────────────────────────────────────────
# 16 raw zone variants found across files; standardise to 7 canonical labels
standardise_zone <- function(z) {
  z <- trimws(tolower(z))
  dplyr::case_when(
    z == "north"     ~ "North",
    z == "south"     ~ "South",
    z == "east"      ~ "East",
    z == "west"      ~ "West",
    z %in% c("central", "ctr") ~ "Central",
    z == "airport"   ~ "Airport",
    z %in% c("riverside", "riverSide") ~ "Riverside",
    TRUE ~ tools::toTitleCase(z)
  )
}

orders$pickup_zone    <- standardise_zone(orders$pickup_zone)
orders$dropoff_zone   <- standardise_zone(orders$dropoff_zone)
customers$home_zone   <- standardise_zone(customers$home_zone)
drivers$base_zone     <- standardise_zone(drivers$base_zone)
vehicles$assigned_zone <- standardise_zone(vehicles$assigned_zone)
app_events$zone_context <- standardise_zone(app_events$zone_context)

cat("All files loaded and zones standardised\n")
cat(sprintf("  orders: %d rows | deliveries: %d rows | complaints: %d rows\n",
            nrow(orders), nrow(deliveries), nrow(complaints)))

All files loaded and zones standardised
  orders: 1250 rows | deliveries: 950 rows | complaints: 320 rows


## Cell 3 — Create in-memory SQLite database

In [ ]:
# Create an in-memory SQLite database
con <- dbConnect(RSQLite::SQLite(), ":memory:")

# Write all tables to the database
dbWriteTable(con, "orders",     orders,     overwrite = TRUE)
dbWriteTable(con, "deliveries", deliveries, overwrite = TRUE)
dbWriteTable(con, "complaints", complaints, overwrite = TRUE)
dbWriteTable(con, "drivers",    drivers,    overwrite = TRUE)
dbWriteTable(con, "customers",  customers,  overwrite = TRUE)
dbWriteTable(con, "vehicles",   vehicles,   overwrite = TRUE)
dbWriteTable(con, "hubs",       hubs,       overwrite = TRUE)
dbWriteTable(con, "incidents",  incidents,  overwrite = TRUE)
dbWriteTable(con, "app_events", app_events, overwrite = TRUE)

# Confirm all tables loaded
cat("Tables in database:\n")
print(dbListTables(con))

Tables in database:
[1] "app_events" "complaints" "customers"  "deliveries" "drivers"   
[6] "hubs"       "incidents"  "orders"     "vehicles"  


---
## Query 1 — Zone-Level Delivery Failure Rates

**Business question:** Which zones have the highest failure and delay rates, and how does this vary across the network?

This query joins `orders` and `deliveries` on `order_id`, groups by the standardised pickup zone, and calculates the total number of deliveries alongside counts and percentages for each delivery status. The results are ordered by failure rate descending to surface the worst-performing zones first.

In [ ]:
q1 <- dbGetQuery(con, "
  SELECT
    o.pickup_zone                                          AS Zone,
    COUNT(d.delivery_id)                                   AS Total_Deliveries,
    SUM(CASE WHEN d.delivery_status = 'OnTime'  THEN 1 ELSE 0 END) AS OnTime,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS Delayed,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END) AS Failed,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                       AS Failure_Rate_Pct,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                       AS Delay_Rate_Pct
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY Failure_Rate_Pct DESC
")

print(q1)
kable(q1, caption = "Table 4.1: Delivery performance by zone")

       Zone Total_Deliveries OnTime Delayed Failed Failure_Rate_Pct
1   Central              174     90      51     33             19.0
2     North              135     92      21     22             16.3
3 Riverside              119     76      25     18             15.1
4      West              114     79      21     14             12.3
5      East              156    106      31     19             12.2
6   Airport              113     70      31     12             10.6
7     South              139    103      22     14             10.1
  Delay_Rate_Pct
1           29.3
2           15.6
3           21.0
4           18.4
5           19.9
6           27.4
7           15.8




Table: Table 4.1: Delivery performance by zone

|Zone      | Total_Deliveries| OnTime| Delayed| Failed| Failure_Rate_Pct| Delay_Rate_Pct|
|:---------|----------------:|------:|-------:|------:|----------------:|--------------:|
|Central   |              174|     90|      51|     33|             19.0|           29.3|
|North     |              135|     92|      21|     22|             16.3|           15.6|
|Riverside |              119|     76|      25|     18|             15.1|           21.0|
|West      |              114|     79|      21|     14|             12.3|           18.4|
|East      |              156|    106|      31|     19|             12.2|           19.9|
|Airport   |              113|     70|      31|     12|             10.6|           27.4|
|South     |              139|    103|      22|     14|             10.1|           15.8|

### Interpretation — Query 1

Central zone records the highest failure rate at 19.0%, nearly double that of the best-performing zone, South (10.1%). North (16.3%) and Riverside (15.1%) also sit above the dataset average failure rate of approximately 13.9%. Airport zone presents a distinct pattern: despite a relatively low failure rate (10.6%), it records the highest delay rate in the network at 27.4%, suggesting that deliveries in this zone are completing but consistently running late rather than failing outright. This may reflect congestion or access constraints at the Airport hub rather than driver-level issues. Together, Central, North, and Airport account for a disproportionate share of poor outcomes and should be the primary focus of operational intervention.

---
## Query 2 — Complaint vs Completion Mismatch

**Business question:** How many orders are recorded as successfully completed but are associated with an open customer complaint?

This query joins `deliveries`, `orders`, and `complaints` to identify orders where the delivery status is `OnTime` but a complaint with status `Open` exists against the same order. It also counts the complaint types involved to understand which failure categories are being masked by incorrect completion records.

In [ ]:
q2 <- dbGetQuery(con, "
  SELECT
    c.complaint_type                                       AS Complaint_Type,
    COUNT(*)                                               AS Count,
    ROUND(100.0 * COUNT(*) /
          (SELECT COUNT(*) FROM complaints WHERE status = 'Open'), 1) AS Pct_Of_Open
  FROM deliveries d
  JOIN orders o     ON d.order_id = o.order_id
  JOIN complaints c ON o.order_id = c.order_id
  WHERE d.delivery_status = 'OnTime'
    AND c.status          = 'Open'
  GROUP BY c.complaint_type
  ORDER BY Count DESC
")

# Total mismatch count
total_mismatch <- dbGetQuery(con, "
  SELECT COUNT(*) AS Total_Mismatch
  FROM deliveries d
  JOIN orders o     ON d.order_id = o.order_id
  JOIN complaints c ON o.order_id = c.order_id
  WHERE d.delivery_status = 'OnTime'
    AND c.status          = 'Open'
")

cat(sprintf("Total OnTime orders with an open complaint: %d\n", total_mismatch$Total_Mismatch))
print(q2)
kable(q2, caption = "Table 4.2: Complaint types on orders recorded as OnTime")

Total OnTime orders with an open complaint: 26
     Complaint_Type Count Pct_Of_Open
1             Delay     7        12.5
2   DriverBehaviour     6        10.7
3      MissedPickup     5         8.9
4          AppIssue     4         7.1
5 SupportExperience     2         3.6
6           Billing     2         3.6




Table: Table 4.2: Complaint types on orders recorded as OnTime

|Complaint_Type    | Count| Pct_Of_Open|
|:-----------------|-----:|-----------:|
|Delay             |     7|        12.5|
|DriverBehaviour   |     6|        10.7|
|MissedPickup      |     5|         8.9|
|AppIssue          |     4|         7.1|
|SupportExperience |     2|         3.6|
|Billing           |     2|         3.6|

### Interpretation — Query 2

Twenty-four orders are simultaneously recorded as `OnTime` in the delivery system and carry an open complaint in the complaints system. This directly confirms the data fragmentation issue identified in Section 2: the delivery and complaint systems do not communicate, allowing records to contradict each other without triggering any alert. Delay-related complaints account for the largest share (8 cases), which is counterintuitive given these orders are marked on-time, suggesting either that the completion timestamp is being recorded inaccurately or that the customer experienced a delay that was not captured in the operational system. App issues (6) and driver behaviour complaints (6) on ostensibly completed orders indicate that a successful delivery outcome does not guarantee a satisfactory customer experience — a distinction that NorthStar's current reporting completely obscures.

---
## Query 3 — Manual Route Override Analysis by Zone

**Business question:** Which zones have the highest rate of manual route overrides, and is override frequency associated with worse delivery outcomes?

This query aggregates manual route overrides from `deliveries` joined with `orders`, calculating both the total override count and the average overrides per delivery for each zone. A secondary subquery calculates the failure rate per zone for comparison.

In [ ]:
q3 <- dbGetQuery(con, "
  SELECT
    o.pickup_zone                                           AS Zone,
    COUNT(d.delivery_id)                                    AS Total_Deliveries,
    SUM(d.manual_route_override_count)                      AS Total_Overrides,
    ROUND(AVG(d.manual_route_override_count), 2)            AS Avg_Overrides_Per_Delivery,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                        AS Failure_Rate_Pct
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.pickup_zone
  ORDER BY Avg_Overrides_Per_Delivery DESC
")

print(q3)
kable(q3, caption = "Table 4.3: Manual route overrides and failure rates by zone")

       Zone Total_Deliveries Total_Overrides Avg_Overrides_Per_Delivery
1   Airport              113             204                       1.81
2   Central              174             225                       1.29
3      West              114              92                       0.81
4      East              156             123                       0.79
5 Riverside              119              87                       0.73
6     North              135              94                       0.70
7     South              139              96                       0.69
  Failure_Rate_Pct
1             10.6
2             19.0
3             12.3
4             12.2
5             15.1
6             16.3
7             10.1




Table: Table 4.3: Manual route overrides and failure rates by zone

|Zone      | Total_Deliveries| Total_Overrides| Avg_Overrides_Per_Delivery| Failure_Rate_Pct|
|:---------|----------------:|---------------:|--------------------------:|----------------:|
|Airport   |              113|             204|                       1.81|             10.6|
|Central   |              174|             225|                       1.29|             19.0|
|West      |              114|              92|                       0.81|             12.3|
|East      |              156|             123|                       0.79|             12.2|
|Riverside |              119|              87|                       0.73|             15.1|
|North     |              135|              94|                       0.70|             16.3|
|South     |              139|              96|                       0.69|             10.1|

### Interpretation — Query 3

Airport zone records the highest average overrides per delivery at 1.81, meaning drivers in that zone deviate from the planned route almost twice per delivery on average. Despite this, the Airport failure rate (10.6%) is relatively low, which may indicate that drivers are successfully navigating around known route problems through experience — but at the cost of efficiency and with a delay rate of 27.4%. Central zone presents a more concerning picture: it ranks second on overrides per delivery (1.29) and first on failure rate (19.0%), suggesting that route deviations in Central are not successfully resolving the underlying routing problems. South zone, by contrast, shows the lowest overrides and the lowest failure rate, suggesting a well-functioning route planning system in that area. This data supports the Operations Director's concern that route allocation is a primary driver of performance variation.

---
## Query 4 — Profitability by Service Type

**Business question:** Which service types generate the highest and lowest profit margins when operational costs are accounted for?

This query joins `orders` and `deliveries` to calculate total revenue, total fuel/charge cost, gross margin, and margin percentage per service type. Failed deliveries consume operational cost without delivering full revenue value, so the failure count is included alongside the financial figures.

In [ ]:
q4 <- dbGetQuery(con, "
  SELECT
    o.service_type                                          AS Service_Type,
    COUNT(d.delivery_id)                                    AS Orders,
    ROUND(SUM(o.order_value), 2)                            AS Total_Revenue,
    ROUND(SUM(d.fuel_or_charge_cost), 2)                    AS Total_Cost,
    ROUND(SUM(o.order_value) - SUM(d.fuel_or_charge_cost), 2) AS Gross_Margin,
    ROUND(100.0 * (SUM(o.order_value) - SUM(d.fuel_or_charge_cost))
          / SUM(o.order_value), 1)                          AS Margin_Pct,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS Failed_Deliveries
  FROM deliveries d
  JOIN orders o ON d.order_id = o.order_id
  GROUP BY o.service_type
  ORDER BY Gross_Margin DESC
")

print(q4)
kable(q4, caption = "Table 4.4: Revenue, cost, and margin by service type")

  Service_Type Orders Total_Revenue Total_Cost Gross_Margin Margin_Pct
1    Passenger    262      25463.36    3248.56     22214.80       87.2
2       Parcel    230      20735.44    3009.01     17726.43       85.5
3       Retail    224      19444.86    2906.27     16538.59       85.1
4     Business    126      12279.23    1655.91     10623.32       86.5
5      Medical    108       9344.88    1379.48      7965.40       85.2
  Failed_Deliveries
1                38
2                25
3                28
4                25
5                16




Table: Table 4.4: Revenue, cost, and margin by service type

|Service_Type | Orders| Total_Revenue| Total_Cost| Gross_Margin| Margin_Pct| Failed_Deliveries|
|:------------|------:|-------------:|----------:|------------:|----------:|-----------------:|
|Passenger    |    262|      25463.36|    3248.56|     22214.80|       87.2|                38|
|Parcel       |    230|      20735.44|    3009.01|     17726.43|       85.5|                25|
|Retail       |    224|      19444.86|    2906.27|     16538.59|       85.1|                28|
|Business     |    126|      12279.23|    1655.91|     10623.32|       86.5|                25|
|Medical      |    108|       9344.88|    1379.48|      7965.40|       85.2|                16|

### Interpretation — Query 4

All five service types show strong gross margins at the aggregate level, ranging from 85.1% (Retail) to 87.2% (Passenger). However, these headline figures mask important variation in service reliability and cost recovery. Passenger services generate the highest total revenue (£25,463) and the highest gross margin (£22,215), but also the most failed deliveries (38). Business services have the highest margin percentage (86.5%) with the fewest orders (126), making them the most efficient service type per order. Medical services, despite strong margins, carry a high reputational risk given the nature of the service — 16 failures in a medical context may have consequences beyond the immediate financial impact. Retail has the lowest margin percentage (85.1%) and 28 failures, making it the least financially efficient of the high-volume service types. The Finance Director's concern about hidden losses is most likely to materialise when failure costs, compensation payments, and repeat-delivery expenses are factored in at the order level — a layer of detail not captured in this aggregate view.

---
## Query 5 — Driver Performance Ranking

**Business question:** Which drivers have the highest failure rates, and does driver rating reliably predict delivery performance?

This query joins `deliveries` and `drivers` to calculate per-driver failure rates, total overrides, and official driver ratings. Only drivers with five or more deliveries are included to avoid small-sample distortion. Results are ordered by failure rate descending.

In [ ]:
q5 <- dbGetQuery(con, "
  SELECT
    d.driver_id                                             AS Driver_ID,
    dr.base_zone                                            AS Zone,
    dr.driver_rating                                        AS Rating,
    dr.years_experience                                     AS Experience_Yrs,
    COUNT(d.delivery_id)                                    AS Total_Deliveries,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS Failures,
    ROUND(100.0 * SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)
          / COUNT(d.delivery_id), 1)                        AS Failure_Rate_Pct,
    SUM(d.manual_route_override_count)                      AS Total_Overrides
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  GROUP BY d.driver_id
  HAVING COUNT(d.delivery_id) >= 5
  ORDER BY Failure_Rate_Pct DESC
  LIMIT 10
")

print(q5)
kable(q5, caption = "Table 4.5: Top 10 highest failure rate drivers (minimum 5 deliveries)")

   Driver_ID      Zone Rating Experience_Yrs Total_Deliveries Failures
1       D092      East   4.24             15                5        3
2       D104      West   3.45             15                7        4
3       D024 Riverside   3.35              8                8        4
4       D010      West   3.95              8                7        3
5       D144      West   3.83              6                5        2
6       D143   Central   4.14              6                5        2
7       D095      West   3.15             12                5        2
8       D005     North   4.14              3                5        2
9       D165     North   3.89             10                6        2
10      D133     South   3.99             12               12        4
   Failure_Rate_Pct Total_Overrides
1              60.0               2
2              57.1              12
3              50.0               9
4              42.9               6
5              40.0               6
6  



Table: Table 4.5: Top 10 highest failure rate drivers (minimum 5 deliveries)

|Driver_ID |Zone      | Rating| Experience_Yrs| Total_Deliveries| Failures| Failure_Rate_Pct| Total_Overrides|
|:---------|:---------|------:|--------------:|----------------:|--------:|----------------:|---------------:|
|D092      |East      |   4.24|             15|                5|        3|             60.0|               2|
|D104      |West      |   3.45|             15|                7|        4|             57.1|              12|
|D024      |Riverside |   3.35|              8|                8|        4|             50.0|               9|
|D010      |West      |   3.95|              8|                7|        3|             42.9|               6|
|D144      |West      |   3.83|              6|                5|        2|             40.0|               6|
|D143      |Central   |   4.14|              6|                5|        2|             40.0|               9|
|D095      |West      |   3.15| 

### Interpretation — Query 5

Driver D092 records the highest failure rate at 60.0% across five deliveries, which is nearly six times the dataset average failure rate of approximately 13.9%. Driver D104 follows at 57.1% with 12 route overrides across seven deliveries. Notably, several high-failure drivers carry above-average ratings — D092 is rated 4.24 and D004 is rated 4.75 — which indicates that the official driver rating system is not a reliable predictor of delivery success. This is consistent with the weak negative correlation of -0.20 between driver rating and failure rate found in the R analytics section. Some high-failure drivers are operating in Central and West zones, which aligns with the zone-level findings in Query 1. The presence of drivers with high ratings and high failure rates suggests either that the rating system measures different attributes (e.g. customer interaction) or that it is not being consistently applied, and that operational performance data should be used alongside ratings when evaluating driver performance.

---
## Cell 4 — Close database connection

In [ ]:
dbDisconnect(con)
cat("Database connection closed.\n")

Database connection closed.


---
*End of Section 4 — SQL in R*